# Customer Segmentation using K-Means Clustering
**Unsupervised Machine Learning | Mall Customer Dataset**

This notebook walks through the complete pipeline:
1. Data Loading & Inspection
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Scaling
5. Optimal K Selection (Elbow Method)
6. K-Means Clustering
7. Cluster Visualization
8. Business Insights

## 1. Import Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/Mall_Customers.csv')

print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
df.head(10)

In [ ]:
# Dataset statistics
df.describe()

## 3. Data Cleaning

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# Check for duplicates
print(f'\nDuplicate rows: {df.duplicated().sum()}')

# Clean
df = df.drop_duplicates().dropna()

# Encode gender
df['Gender_Encoded'] = df['Gender'].map({'Male': 0, 'Female': 1})

print(f'\nClean dataset shape: {df.shape}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- Gender Distribution ---
fig, ax = plt.subplots(figsize=(6, 5))
counts = df['Gender'].value_counts()
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['#5B8DB8', '#E88D8D'], startangle=140, textprops={'fontsize': 12})
ax.set_title('Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Distributions ---
cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cols):
    sns.histplot(df[col], kde=True, ax=ax, color='#5B8DB8', edgecolor='white')
    ax.set_title(f'{col}\nDistribution', fontweight='bold')
fig.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- Boxplots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cols):
    sns.boxplot(y=df[col], ax=ax, color='#98D8C8')
    ax.set_title(f'{col} Boxplot', fontweight='bold')
fig.suptitle('Boxplots — Outlier Check', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap ---
num_df = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Gender_Encoded']]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Income vs Spending Score Scatter ---
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'],
                     c=df['Gender_Encoded'], cmap='coolwarm', alpha=0.7, s=70)
ax.set_xlabel('Annual Income (k$)', fontsize=12)
ax.set_ylabel('Spending Score (1-100)', fontsize=12)
ax.set_title('Income vs Spending Score', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Gender (0=Male, 1=Female)')
plt.tight_layout()
plt.show()

## 5. Feature Selection & Scaling

In [ ]:
# Select clustering features
FEATURE_COLS = ['Annual Income (k$)', 'Spending Score (1-100)', 'Age']
X = df[FEATURE_COLS].values

# Standardize — K-Means is distance-based so scale matters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X_scaled.shape)
print('Mean after scaling (should be ~0):', X_scaled.mean(axis=0).round(3))
print('Std after scaling  (should be ~1):', X_scaled.std(axis=0).round(3))

## 6. Elbow Method — Optimal K

In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 11), wcss, marker='o', color='#E88D8D', linewidth=2, markersize=8)
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.6, label='Elbow at K=5')
ax.set_title('Elbow Method — Optimal K Selection', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('\nThe curve bends (elbow) most sharply at K=5, indicating 5 is the optimal number of clusters.')

## 7. Apply K-Means Clustering (K=5)

In [ ]:
OPTIMAL_K = 5
km = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=10, random_state=42)
km.fit(X_scaled)

df['Cluster'] = km.labels_

sil_score = silhouette_score(X_scaled, km.labels_)
print(f'Silhouette Score: {sil_score:.4f}')
print(f'Inertia (WCSS): {km.inertia_:.2f}')
print('\nCluster distribution:')
print(df['Cluster'].value_counts().sort_index())

## 8. Segment Labeling

In [ ]:
summary = df.groupby('Cluster')[FEATURE_COLS].mean().round(2)
print('Cluster Centroids (original scale):')
print(summary)

# Assign human-readable labels based on income & spending
def label_cluster(row):
    income = row['Annual Income (k$)']
    score  = row['Spending Score (1-100)']
    if income >= 70 and score >= 60:
        return 'Premium Customers'
    elif income >= 70 and score < 45:
        return 'Careful Spenders'
    elif income < 45 and score >= 55:
        return 'High Spenders'
    elif income < 45 and score < 45:
        return 'Budget Customers'
    else:
        return 'Average Customers'

labels = {cluster_id: label_cluster(row) for cluster_id, row in summary.iterrows()}
df['Segment'] = df['Cluster'].map(labels)

print('\nSegment mapping:', labels)
print('\nCustomers per segment:')
print(df['Segment'].value_counts())

## 9. Cluster Visualizations

In [ ]:
# --- 2D Scatter — Income vs Spending Score ---
palette = sns.color_palette('Set2', df['Segment'].nunique())
color_map = dict(zip(sorted(df['Segment'].unique()), palette))

fig, ax = plt.subplots(figsize=(11, 7))
for seg in sorted(df['Segment'].unique()):
    subset = df[df['Segment'] == seg]
    ax.scatter(subset['Annual Income (k$)'], subset['Spending Score (1-100)'],
               label=seg, color=color_map[seg], alpha=0.75, edgecolors='white', s=90)

# Centroids
centers = df.groupby('Segment')[['Annual Income (k$)', 'Spending Score (1-100)']].mean()
ax.scatter(centers['Annual Income (k$)'], centers['Spending Score (1-100)'],
           s=280, marker='X', color='black', zorder=5, label='Centroids')

ax.set_xlabel('Annual Income (k$)', fontsize=13)
ax.set_ylabel('Spending Score (1-100)', fontsize=13)
ax.set_title('Customer Segments — K-Means Clustering', fontsize=15, fontweight='bold')
ax.legend(title='Segment', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- Segment Profiles ---
seg_summary = df.groupby('Segment')[FEATURE_COLS].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, FEATURE_COLS):
    sns.barplot(data=seg_summary, x='Segment', y=col, hue='Segment',
                palette='Set2', legend=False, ax=ax)
    ax.set_title(f'Avg {col}', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('Average Feature Values per Segment', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- Segment Size ---
counts = df['Segment'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index,
            palette=sns.color_palette('Set2', len(counts)), legend=False, ax=ax)
ax.set_title('Number of Customers per Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Segment')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Segment Summary Table

In [ ]:
final_summary = df.groupby('Segment')[FEATURE_COLS].mean().round(2)
final_summary['Customer Count'] = df['Segment'].value_counts()
final_summary.style.background_gradient(cmap='YlOrRd')

## 11. Business Insights

| Segment | Profile | Recommendation |
|---|---|---|
| **Premium Customers** | High income, high spending, young | VIP programs, early access, personalized offers |
| **Careful Spenders** | High income, low spending, older | Exclusive discounts, savings messaging, ROI-focused campaigns |
| **Budget Customers** | Low income, low spending | Flash sales, bundles, loyalty points |
| **High Spenders** | Low income, high spending | Flexible payments, trend-driven marketing |
| **Average Customers** | Mid income, mid spending | Seasonal campaigns, upsell nudges |

**Key takeaway**: Premium Customers drive revenue; Careful Spenders are the biggest untapped opportunity.

In [ ]:
# Save segmented dataset
df.to_csv('../data/customers_segmented.csv', index=False)
print('Segmented dataset saved to data/customers_segmented.csv')
print(f'Final dataset shape: {df.shape}')